In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import root_mean_squared_error
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestRegressor

from combine_features import read_data

# Experiments

In [35]:
df = read_data("../rhea-soil-nutrient-prediction-challenge/Train.csv")
df.head()

,ID,Longitude,Latitude,Depth_cm,ph,Area,Cropland nitrogen per unit area,Cropland phosphorus per unit area,Cropland potassium per unit area,tmin_avg,...,Cu,Fe,Mg,Mn,N,P,K,Na,S,Zn
0,BF9XTB,37.65189,-3.15440,20-50,6.405,Kenya,14.202173,3.902236,-7.263155,15.240530,...,5.826,81.780,306.836,270.240,0.79,NaN,300.951,NaN,NaN,NaN
1,2RWYTR,37.63612,-3.08585,20-50,6.419,Kenya,14.202173,3.902236,-7.263155,15.240530,...,4.346,97.198,407.980,185.557,1.11,NaN,292.696,NaN,NaN,NaN
2,XZI9Q6,39.55580,-2.67218,20-50,8.388,Kenya,14.202173,3.902236,-7.263155,21.969696,...,3.657,42.672,1256.319,178.299,0.45,NaN,814.911,NaN,NaN,NaN
3,4CBCVY,39.55477,-2.67196,20-50,8.302,Kenya,14.202173,3.902236,-7.263155,21.969696,...,3.376,52.861,1322.732,464.137,0.31,NaN,815.337,NaN,NaN,NaN
4,F9GK9S,39.55477,-2.67196,20-50,8.292,Kenya,14.202173,3.902236,-7.263155,21.969696,...,3.351,46.057,1134.898,274.565,0.45,NaN,928.238,NaN,NaN,NaN


In [57]:
df.iloc[:, 40:].info()

<class 'pandas.DataFrame'>
RangeIndex: 13454 entries, 0 to 13453
Data columns (total 14 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   Ca         13454 non-null  float64
 1   C_organic  13454 non-null  float64
 2   C_total    13416 non-null  float64
 3   Cu         13416 non-null  float64
 4   Fe         13416 non-null  float64
 5   Mg         13454 non-null  float64
 6   Mn         13416 non-null  float64
 7   N          13416 non-null  float64
 8   P          0 non-null      float64
 9   K          13454 non-null  float64
 10  Na         38 non-null     float64
 11  S          0 non-null      float64
 12  Zn         0 non-null      float64
 13  Depth      13454 non-null  int64  
dtypes: float64(13), int64(1)
memory usage: 1.4 MB


In [36]:
pred_to_keep = pd.read_csv("../rhea-soil-nutrient-prediction-challenge/TargetPred_To_Keep.csv")
pred_to_keep.head()

,ID,Al,B,Ca,Cu,Fe,K,Mg,Mn,N,Na,P,S,Zn
0,00A83S,1,1,1,1,1,1,1,1,1,1,1,1,1
1,00F2Q3,1,1,1,1,1,1,1,1,1,1,1,1,1
2,00FNFP,1,0,1,1,1,1,1,1,1,0,0,0,0
3,01MFSS,1,0,1,1,1,1,1,1,1,0,0,0,0
4,02851F,1,0,1,1,1,1,1,1,1,0,0,0,0


In [37]:
pred_to_keep.sum()

ID    00A83S00F2Q300FNFP01MFSS02851F02C3Q502I9O502L2...
Al                                                 6070
B                                                  1345
Ca                                                 6070
Cu                                                 6065
Fe                                                 6065
K                                                  6070
Mg                                                 6070
Mn                                                 6065
N                                                  6065
Na                                                 1350
P                                                  1345
S                                                  1345
Zn                                                 1345
dtype: object

In [38]:
encoder = LabelEncoder()
df['Depth'] = encoder.fit_transform(df['Depth_cm'])
# df.drop(columns=['ID', 'Area', 'Depth_cm'], inplace=True)
# df.dropna(axis=1, thresh=int(0.8 * df.shape[0]), inplace=True) # drops target columns :(
# df.dropna(axis=0, inplace=True)
df.head()

,ID,Longitude,Latitude,Depth_cm,ph,Area,Cropland nitrogen per unit area,Cropland phosphorus per unit area,Cropland potassium per unit area,tmin_avg,...,Fe,Mg,Mn,N,P,K,Na,S,Zn,Depth
0,BF9XTB,37.65189,-3.15440,20-50,6.405,Kenya,14.202173,3.902236,-7.263155,15.240530,...,81.780,306.836,270.240,0.79,NaN,300.951,NaN,NaN,NaN,1
1,2RWYTR,37.63612,-3.08585,20-50,6.419,Kenya,14.202173,3.902236,-7.263155,15.240530,...,97.198,407.980,185.557,1.11,NaN,292.696,NaN,NaN,NaN,1
2,XZI9Q6,39.55580,-2.67218,20-50,8.388,Kenya,14.202173,3.902236,-7.263155,21.969696,...,42.672,1256.319,178.299,0.45,NaN,814.911,NaN,NaN,NaN,1
3,4CBCVY,39.55477,-2.67196,20-50,8.302,Kenya,14.202173,3.902236,-7.263155,21.969696,...,52.861,1322.732,464.137,0.31,NaN,815.337,NaN,NaN,NaN,1
4,F9GK9S,39.55477,-2.67196,20-50,8.292,Kenya,14.202173,3.902236,-7.263155,21.969696,...,46.057,1134.898,274.565,0.45,NaN,928.238,NaN,NaN,NaN,1


In [71]:
target = [
        "Al",
        "B",
        "Ca",
        "C_organic",
        "C_total",
        "Cu",
        "Fe",
        "K",
        "Mg",
        "Mn",
        "N",
        "Na",
        "P",
        "S",
        "Zn",
    ]
x = df.drop(columns=target, errors='ignore')
x = x.drop(columns=['ID', 'Area', 'Depth_cm'], errors='ignore')
y_columns = df.columns.difference(x.columns)
y_columns = [col for col in y_columns if col not in ["C_organic", "C_total", "Area", "Depth_cm", "ID"]]
y = df[y_columns]
y.fillna(0, inplace=True)
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

In [72]:
x_train.head()

,Longitude,Latitude,ph,Cropland nitrogen per unit area,Cropland phosphorus per unit area,Cropland potassium per unit area,tmin_avg,tmax_avg,prec_avg,Bananas,...,Rice,"Seed cotton, unginned",Sesame seed,Sorghum,Soya beans,Sugar cane,Sweet potatoes,Tomatoes,Unmanufactured tobacco,Depth
4721,-2.19999,7.20972,6.969,-2.917073,-2.796745,-18.097209,22.215910,30.598484,103.590550,10390.800000,...,2719.254545,900.927273,0.000000,1169.454545,1634.036364,25018.527273,1781.009091,7377.254545,413.118182,0
7732,-11.01100,14.76583,6.699,3.931445,-2.550627,-3.758418,23.316288,37.174244,54.082947,34266.136364,...,3123.272727,985.490909,420.227273,1037.981818,620.363636,72963.909091,17452.418182,17080.290909,2304.281818,0
5746,-1.07758,7.64100,6.509,-2.917073,-2.796745,-18.097209,22.660984,32.393940,103.886536,10390.800000,...,2719.254545,900.927273,0.000000,1169.454545,1634.036364,25018.527273,1781.009091,7377.254545,413.118182,0
12178,14.92311,-16.19789,6.161,9.219964,0.017264,-9.459173,15.484848,30.342804,61.349070,24064.809091,...,1022.400000,1446.963636,263.136364,227.736364,603.709091,38876.818182,8256.936364,10544.109091,1018.445455,0
12952,15.22342,-12.29724,5.903,9.219964,0.017264,-9.459173,13.121212,25.611742,121.739590,24064.809091,...,1022.400000,1446.963636,263.136364,227.736364,603.709091,38876.818182,8256.936364,10544.109091,1018.445455,0


In [73]:
y_train.head()

,Al,B,Ca,Cu,Fe,K,Mg,Mn,N,Na,P,S,Zn
4721,668.664,0.0,2440.023,3.406,141.737,138.646,286.859,337.732,1.90,0.0,0.0,0.0,0.0
7732,981.937,0.0,2336.159,2.910,159.714,271.761,671.278,93.069,0.56,0.0,0.0,0.0,0.0
5746,837.742,0.0,2120.437,2.700,109.035,218.283,327.699,250.103,1.91,0.0,0.0,0.0,0.0
12178,730.772,0.0,740.481,0.966,172.359,87.640,134.619,102.362,0.80,0.0,0.0,0.0,0.0
12952,1145.199,0.0,541.405,2.794,83.008,160.257,149.018,69.130,0.62,0.0,0.0,0.0,0.0


In [74]:
models = {}
for col in y_columns:
    model = RandomForestRegressor(n_estimators=100, random_state=42)
    model.fit(x_train, y_train[col])
    y_pred = model.predict(x_test)
    rmse = root_mean_squared_error(y_test[col], y_pred)
    models[col] = model
    print(f"{col}: RMSE = {rmse}")

Al: RMSE = 119.83690123975009
B: RMSE = 0.0
Ca: RMSE = 860.4876479192877
Cu: RMSE = 0.7373721257800218
Fe: RMSE = 26.40708132091743
K: RMSE = 95.01308548898767
Mg: RMSE = 134.06745534393096
Mn: RMSE = 41.544999694051576
N: RMSE = 0.42885104413918784
Na: RMSE = 1.3454649259216942
P: RMSE = 0.0
S: RMSE = 0.0
Zn: RMSE = 0.0


In [75]:
for col, model in models.items():
    fi = pd.DataFrame(model.feature_importances_, index=x_train.columns, columns=['Feature Importance'])
    fi.sort_values(by='Feature Importance', ascending=False, inplace=True)
    print(f"Feature importance for {col}:")
    print(fi.head(10))

Feature importance for Al:
                                                    Feature Importance
prec_avg                                                      0.533798
Longitude                                                     0.109685
ph                                                            0.101421
Latitude                                                      0.073544
tmin_avg                                                      0.047903
Soya beans                                                    0.047212
tmax_avg                                                      0.022420
Pineapples                                                    0.020706
Depth                                                         0.006732
Chillies and peppers, green (Capsicum spp. and ...            0.005770
Feature importance for B:
                                Feature Importance
Longitude                                      0.0
Latitude                                       0.0
Oranges      

# Generating test answers

In [76]:
test_df = read_data("../rhea-soil-nutrient-prediction-challenge/TestSet.csv")
test_df.head()

KeyError: "['start_date', 'end_date', 'horizon_lower', 'horizon_upper', 'electrical_conductivity'] not found in axis"

In [20]:
test_df.shape

(6070, 17)